In [ ]:
from datetime import datetime
from pathlib import Path
import sys

sys.path.insert(0, str(Path("../..").resolve()))

import ad_safe_lib as ad_safe

print("challenge:", ad_safe.CHALLENGE_DIR)
print("device:", ad_safe.DEVICE)
print("dataset:", ad_safe.DATA_DIR)
examples_root = ad_safe.AD_SAFE_RUNS_DIR / "notebook_examples"
examples_root.mkdir(parents=True, exist_ok=True)


# Tiny Sweep And Metrics Comparison

Build a few jobs programmatically, run the shared training runner, then compare models with the shared evaluation runner.


In [ ]:
run_id = "notebook-04-" + datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
output_dir = examples_root / run_id

# Slightly larger subsets reduce variance while keeping notebook runtime manageable.
train_source = ad_safe.DatasetSourceSpec("train", fraction=0.08, seed=40)
eval_sources = {
    "val": ad_safe.DatasetSourceSpec("val", fraction=0.10, seed=101),
    "test": ad_safe.DatasetSourceSpec("test", fraction=0.10, seed=102),
}

# Include one compact pretrained baseline for comparison.
backbones = ["simple_mlp", "simple_cnn", "mobilenet_v3_small"]

backbone_settings = {
    "simple_mlp": {"learning_rate": 8e-4, "epochs": 8, "patience": 3, "batch_size": 16},
    "simple_cnn": {"learning_rate": 4e-4, "epochs": 8, "patience": 3, "batch_size": 16},
    "mobilenet_v3_small": {"learning_rate": 8e-4, "epochs": 3, "patience": 2, "batch_size": 16},
}

jobs = []
for job_index, backbone in enumerate(backbones):
    settings = backbone_settings[backbone]

    # Scratch simple backbones must train end-to-end, not classifier-only.
    train_all_layers = backbone.startswith("simple_")

    phase = ad_safe.PhaseSpec(
        job_index=job_index,
        phase_index=0,
        prefix=backbone,
        title="quick",
        requested_seed=40 + job_index,
        config=ad_safe.TrainingConfig(
            base_model=backbone,
            epochs=settings["epochs"],
            patience=settings["patience"],
            batch_size=settings["batch_size"],
            learning_rate=(settings["learning_rate"],),
            resplit_runs=1,
        ),
        unfreeze_all=train_all_layers,
        model_filename=f"{backbone}.pt",
        history_filename=f"{backbone}_history.png",
        json_filename=f"{backbone}.json",
    )
    jobs.append(
        ad_safe.JobSpec(
            job_index=job_index,
            job_id=backbone,
            title=backbone,
            backbone=backbone,
            phases=(phase,),
        )
    )

plan = ad_safe.RunPlan(
    output_dir=output_dir,
    run_id=run_id,
    train_split="train",
    eval_splits=("val", "test"),
    jobs=tuple(jobs),
    resume=False,
    force=True,
    setup_path=output_dir / "setup.json",
    check_foreign_contract=False,
    train_source=train_source,
    eval_sources=eval_sources,
)

result = ad_safe.run_training_plan(plan)
result.metrics_csv_path


In [ ]:
ad_safe.run_evaluation_plan(
    ad_safe.EvaluationPlan(
        models=tuple(
            ad_safe.ModelEvalSpec(path=phase.model_path, title=phase.prefix)
            for phase in result.phase_results
        ),
        datasets=(
            ad_safe.DatasetEvalSpec(name="val", batch_size=8, source=eval_sources["val"]),
            ad_safe.DatasetEvalSpec(name="test", batch_size=8, source=eval_sources["test"]),
        ),
        output_dir=output_dir,
        sort_key="acc_test",
        title="Sweep comparison",
    )
)
